In [6]:
!pip install -q langchain langchain-community langchain-core langchain-pinecone \
                sentence-transformers pypdf pinecone-client

In [10]:
!pip install -q langchain-huggingface

In [1]:
!pip install -q langchain-pinecone

In [11]:
import os
from google.colab import drive
from dotenv import load_dotenv

drive.mount('/content/drive')

# Load .env file (recommended)
load_dotenv('/content/drive/MyDrive/Rag-MCQ-App/.env')

# === SET YOUR KEYS HERE ===
os.environ["XAI_API_KEY"] = "xai-bzNhC4qaCJilX7xrJPbrqzip19wvcJEJDchIIBg5oUys1iyhanRIxG3KAQO0glyHnIE467Hvdm6i0Aw5"        # ← Change
os.environ["PINECONE_API_KEY"] = "pcsk_3aQozQ_3Tiu7rvNK1fFYTC1Vk7UpgnFQLeDJJ3VQbuwps85biEWjV7ucQVZ6PL1VtN9Svf" # ← Change

print("✅ Keys loaded!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Keys loaded!


In [14]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
import json
from transformers import pipeline



In [15]:
# ====================== CONFIG ======================
DATA_PATH = "/content/drive/MyDrive/Rag-MCQ-App/data"
INDEX_NAME = "mcq-bot-index"

# ====================== LOAD DOCUMENTS ======================
loader = PyPDFDirectoryLoader(DATA_PATH)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
texts = text_splitter.split_documents(documents)

print(f"✅ Loaded {len(texts)} chunks")

✅ Loaded 505 chunks


In [16]:
# ====================== EMBEDDINGS + PINECONE ======================
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = PineconeVectorStore.from_documents(
    documents=texts,
    embedding=embeddings,
    index_name=INDEX_NAME
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
print("✅ Pinecone Ready!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Pinecone Ready!


In [21]:
# ====================== FAST LLM ======================
pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    device="cpu",
    max_new_tokens=400,
    temperature=0.4,
    do_sample=True,
    return_full_text=False,
    max_length=None
)

llm = HuggingFacePipeline(pipeline=pipe)



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [22]:
# ====================== STRONGER PROMPT ======================
prompt_template = PromptTemplate(
    input_variables=["context", "topic"],
    template="""
You are a professional exam creator. Generate **exactly one** high-quality multiple choice question.

Context:
{context}

Topic: {topic}

Instructions:
- Create only ONE question
- 4 options (A, B, C, D)
- One correct answer
- Return **ONLY** valid JSON. No other text.

Output format:
{{
  "question": "Write the question here?",
  "options": ["A. First option", "B. Second option", "C. Third option", "D. Fourth option"],
  "answer": "B",
  "explanation": "Brief explanation why the answer is correct"
}}
"""
)

In [ ]:

# ====================== IMPROVED GENERATE FUNCTION ======================
def generate_mcq(topic):
    print(f"⏳ Generating MCQ for: {topic} ...")

    docs = retriever.invoke(topic)
    context = "\n\n".join([doc.page_content for doc in docs])[:2000]

    prompt = prompt_template.format(context=context, topic=topic)
    response = llm.invoke(prompt)

    try:
        # Clean and extract JSON
        text = response.strip()
        # Find the first { and last }
        start = text.find('{')
        end = text.rfind('}') + 1
        if start == -1 or end == 0:
            raise ValueError("No JSON found")

        json_str = text[start:end]
        mcq = json.loads(json_str)

        print("\n" + "="*80)
        print("✅ MCQ GENERATED SUCCESSFULLY")
        print("="*80)
        print(f"**Question:** {mcq['question']}\n")
        for opt in mcq.get("options", []):
            print(opt)
        print(f"\n**Correct Answer:** {mcq.get('answer')}")
        print(f"**Explanation:** {mcq.get('explanation', 'No explanation')}")
        print("="*80)

    except Exception as e:
        print("❌ Failed to parse JSON")
        print("Raw output (first 800 chars):")
        print(response[:800])

# ====================== INTERACTIVE ======================
from IPython.display import clear_output

print("🚀 **Fast MCQ Generator Ready!**\n")
while True:
    clear_output(wait=True)
    topic = input("Enter topic (or 'exit'): ")
    if topic.lower() in ['exit', 'quit']:
        print("Goodbye!")
        break
    if topic.strip():
        generate_mcq(topic)
        input("\nPress Enter for next MCQ...")

Enter topic (or 'exit'): transformer
⏳ Generating MCQ for: transformer ...

✅ MCQ GENERATED SUCCESSFULLY
**Question:** What is the main objective of the transformer model?

To generate text
To classify text
To solve NLP problems
To perform NLP tasks

**Correct Answer:** To generate text
**Explanation:** The transformer model is a powerful architecture for NLP tasks
